In [1]:
# importing necessary libraries
import pandas as pd
import re
import numpy as np
import itertools  # For working with iterators

import torch  # PyTorch library for deep learning
from transformers import AutoModel, AutoTokenizer  # Transformers library for natural language processing
from transformers import TextDataset, LineByLineTextDataset, DataCollatorForLanguageModeling,pipeline, Trainer, TrainingArguments, DataCollatorWithPadding
from transformers import AutoModelForSequenceClassification
from sklearn.utils.class_weight import compute_class_weight

import evaluate
from datasets import Dataset, Image, ClassLabel  # Import custom 'Dataset', 'ClassLabel', and 'Image' classes
from transformers import pipeline  # Transformers library for pipelines
from bs4 import BeautifulSoup  # For parsing HTML content

from sklearn.metrics import (  # Import various metrics from scikit-learn
    accuracy_score,  # For calculating accuracy
    roc_auc_score,  # For ROC AUC score
    confusion_matrix,  # For confusion matrix
    classification_report,  # For classification report
    f1_score  # For F1 score
)

from tqdm import tqdm  # For displaying progress bars
tqdm.pandas()  # Enable progress bars for pandas operations


RuntimeError: Failed to import transformers.models.auto.modeling_auto because of the following error (look up to see its traceback):
Failed to import transformers.generation.utils because of the following error (look up to see its traceback):
cannot import name 'DEFAULT_CIPHERS' from 'urllib3.util.ssl_' (C:\Users\Bagdo\anaconda3\Lib\site-packages\urllib3\util\ssl_.py)

**Data Prep**

In [2]:
dataset_1 = pd.read_csv("data/combined_data.csv")

In [3]:
dataset_1.head(5)

,label,text
0,1,ounce feather bowl hummingbird opec moment ala...
1,1,wulvob get your medircations online qnb ikud v...
2,0,computer connection from cnn com wednesday es...
3,1,university degree obtain a prosperous future m...
4,0,thanks for all your answers guys i know i shou...


In [ ]:
dataset_1.head(4)

In [ ]:
dataset_1['label'].value_counts()

In [ ]:
dataset_1['text']

In [ ]:
dataset_1.rename({'label':'spam','text':'message'}, axis = 1, inplace = True)

In [ ]:
dataset_2 = pd.read_csv("data/final_dataset.csv")

In [ ]:
dataset_2 = dataset_2[['message','spam']]

In [ ]:
dataset_2[:3]

In [ ]:
dataset_1[:3]

In [ ]:
full_dataset = pd.concat([dataset_1, dataset_2])

In [ ]:
full_dataset['spam'].value_counts()

In [ ]:
full_dataset.isnull().sum()

In [ ]:
df = full_dataset.copy()

In [ ]:
df = df[~df['message'].isnull()]

In [ ]:
# Converts Pandas DataFrame (df) into a Hugging Face Dataset object.
dataset = Dataset.from_pandas(df)

In [ ]:
labels_list = sorted(df['spam'].unique())  # [0,1]
class_labels = ClassLabel(num_classes=len(labels_list), names=[str(l) for l in labels_list])

# Since your labels are already integers, no mapping is needed
# You can just cast the column
dataset = dataset.cast_column('spam', class_labels)

In [ ]:
# Splitting the dataset into training and testing sets using the predefined train/test split ratio.
dataset = dataset.train_test_split(test_size=0.2, shuffle=True, stratify_by_column="spam")

# Extracting the training data from the split dataset.
df_train = dataset['train']

# Extracting the testing data from the split dataset.
df_test = dataset['test']

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("roberta-base", use_fast=True, low_cpu_mem_usage=False)

In [ ]:
def preprocess_function(examples):
    return tokenizer(examples["message"], truncation=True)

df_train = df_train.map(preprocess_function, batched=True)
df_test = df_test.map(preprocess_function, batched=True)

In [ ]:
df_train = df_train.remove_columns(['message','__index_level_0__'])
df_test = df_test.remove_columns(['message','__index_level_0__'])

In [ ]:
df_train

**Loading and training model**

In [ ]:
# Load a pre-trained BERT-based model for sequence classification.
model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=2  # only 0 or 1
)
# Calculate and print the number of trainable parameters in millions for the model.
print(model.num_parameters(only_trainable=True) / 1e6)

In [ ]:
# Import the 'load_metric' function from the Hugging Face datasets library to load a metric.
metric = evaluate.load("accuracy")


def compute_metrics(eval_pred):
    # Unpack the 'eval_pred' tuple into 'logits' (predicted logits) and 'labels' (true labels).
    logits, labels = eval_pred
    
    # Calculate the model's predictions by selecting the class with the highest logit value.
    predictions = np.argmax(logits, axis=-1)
    
    # Use the imported metric to compute the accuracy of the model's predictions.
    accuracy = metric.compute(predictions=predictions, references=labels)
    
    # Return the computed accuracy as the evaluation metric.
    return accuracy

In [ ]:
# Your labels are already integers, e.g. 0 = not spam, 1 = spam
classes = np.unique(df['spam'])

# Compute weights for each class automatically
weights = compute_class_weight(class_weight='balanced', classes=classes, y=df['spam'])

# Convert to dictionary for reference (optional)
class_weights = dict(zip(classes, weights))

print("Class weights:", class_weights)

# Ordered weights as a list (aligned with class order 0 → 1)
ordered_weights = [class_weights[c] for c in sorted(class_weights.keys())]
print("Ordered weights:", ordered_weights)

In [ ]:
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.pop("labels")
        # forward pass
        outputs = model(**inputs)
        logits = outputs.get("logits")
        # compute custom loss (suppose one has labels with different weights)
        loss_fct = torch.nn.CrossEntropyLoss(weight=torch.tensor(ordered_weigths, device=model.device).float())
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

In [ ]:
# Create TrainingArguments to configure the training process
training_args = TrainingArguments(
    output_dir='./models',
    logging_dir='./models',
    num_train_epochs=1,  
    per_device_train_batch_size=8,  # Batch size for training dat
    per_device_eval_batch_size=64,  # Batch size for evaluation data
    logging_strategy='steps',  # Logging frequency during training (steps or epoch)
    logging_first_step=True,  # Log the first training step
    load_best_model_at_end=True,  # Load the best model at the end of training
    logging_steps=1,  # Log every training step (useful for debugging)
    learning_rate=3e-6, # Set the learning rate for the optimizer.
    evaluation_strategy='epoch',  # Evaluation frequency (epoch or steps)
    warmup_steps=50,  # Number of warmup steps for the learning rate
    weight_decay=0.02,  # Weight decay for regularization
    eval_steps=1,  # Evaluate every training step (useful for debugging)
    save_strategy='epoch',  # Save model checkpoints every epoch
    save_total_limit=1,  # Limit the number of saved checkpoints to save space
    report_to="mlflow",  # Log training metrics to MLflow
)

In [ ]:
import  accelerate
print(accelerate.__version__)
